[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ChrisW09/Python-for-AI-Driven-Automation/blob/main/fast_track/10_ai_workflows.ipynb)

# 📓 Notebook 10 (fast track) — AI-Assisted Workflows: LLMs, Prompts, and Automation

> **Module:** AI Engineering · **Estimated time:** 60–90 min · **Difficulty:** Intermediate

This notebook is the bridge between everything you've learned and what *modern* AI-driven work looks like. You will:

- Treat a Large Language Model (LLM) like any other Python function.
- Write prompts that are *reliable*, not just clever.
- Get **structured output** out of an LLM so the rest of your pipeline can use it.
- Classify and summarise a batch of real-looking customer feedback.
- Build a tiny **retrieval** workflow that grounds an LLM in your own documents.
- Put it all together in a small end-to-end automation pipeline.

> 📨 **Our running example: the overnight feedback inbox.** Imagine you run a small SaaS business. Every night, customers leave a pile of messages — praise, bug reports, refund demands, feature wishes. By morning *someone* has to read them all, sort them, and write a summary for the team. That someone is about to become a Python script. We'll build that pipeline up piece by piece, and by the end you'll feed it raw text and get back a tidy table of sentiments, topics, and a cost report.

> 🧭 **Mental model — the LLM is a brilliant but forgetful intern.** Picture a sharp new intern who reads and writes beautifully but has **no memory between tasks**, does *exactly* what the note in front of them says (no more, no less), and occasionally hands back sloppy work. You manage them the way you'd manage any intern: write **clear instructions** (the prompt), give them **everything they need on the page** (because there's no memory), ask for output in a **format you can file** (structured JSON), and **spot-check the results** (evaluation). Keep this intern in mind — every section below is really about managing them better.

## 🔌 Runs entirely offline

This notebook uses a **`MockLLM`** that returns deterministic, sensible answers locally — so you can run every cell with **no API key and no internet access**. At the very end you'll see how to swap the mock for a real OpenAI or Anthropic client; the calling code is identical.

## 🎯 Learning objectives

By the end of this notebook you can:

1. Explain what happens when you "call an LLM" — it's just a function call.
2. Use the **system / user / assistant** message structure correctly.
3. Apply the four core prompt patterns: **instructions**, **few-shot**, **structured output**, **chain-of-thought**.
4. Parse JSON output safely with `try / except`.
5. Classify and summarise a batch of records and report the results.
6. Build a tiny retrieval workflow (keyword-based RAG).
7. Reason about **cost, latency, and evaluation** for an AI feature.

## ✅ Prerequisites

Notebooks 1–7 — or at least 1–4 plus comfort with functions, dictionaries, and list comprehensions.

> 🏎️ **You're on the fast track.** This is a trimmed version of the canonical [`06_ai_engineering/22_ai_workflows.ipynb`](../06_ai_engineering/22_ai_workflows.ipynb) — AI workflows. The deepest Stretch exercises (A and B) and the 🎁 Bonus mini-project have been removed to keep the notebook under ~90 minutes; the harder Stretch C and D are kept. Open the canonical version once you want the deeper material.

---


## 1. An LLM call is just a function call

Before we automate that overnight inbox, we need to demystify the thing doing the reading. Forget the marketing copy for a moment. From your code's point of view, calling a Large Language Model looks like this:

```
response = llm(messages=[...], model="...", temperature=...)
```

You hand it a list of **messages** and some settings, and you get back a **string** (plus some metadata). The "intelligence" is on the other side of the wire. *Everything you'll do in this notebook is plumbing around that one function call.*

### Why this perspective matters

If you treat the LLM as magic, your code will be brittle: prompt strings buried in business logic, no error handling, no tests. If you treat it as a *function* — a way to hand your intern a note and get a reply — you get the same engineering hygiene you'd apply to any third-party API: clear inputs, structured outputs, retries on failure, costs you can measure.

## 2. Our offline `MockLLM`

To keep this notebook runnable everywhere, we will use a small `MockLLM` class that imitates a real LLM. It is **not** intelligent — it follows hard-coded rules to produce plausible answers. That is more than enough to learn the *patterns*; you'll swap in a real model at the end.

Run the cell below first — every later cell uses this `llm` object.

In [ ]:
# MockLLM lives in llm_providers.py so every AI notebook uses the same one.
# Swap it for any real provider by changing this one line — the rest of the
# notebook stays identical:
#
#     from llm_providers import OpenAILLM as LLM      # needs `pip install openai`
#     from llm_providers import AnthropicLLM as LLM   # needs `pip install anthropic`
#     from llm_providers import GoogleLLM as LLM      # needs `pip install google-generativeai`
#     from llm_providers import OllamaLLM as LLM      # local — needs Ollama running
import json, re, random, time
from textwrap import shorten

# Make the repo-root llm_providers.py importable no matter which folder
# Jupyter launched from (it lives one level up, at the course root).
import sys, pathlib
_root = next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
              if (p / "llm_providers.py").exists()), None)
if _root:                       # local checkout — use the repo's module
    sys.path.insert(0, str(_root))
else:                           # Colab / standalone — fetch it next to the notebook
    import urllib.request
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/ChrisW09/Python-for-AI-Driven-Automation"
        "/main/llm_providers.py", "llm_providers.py")
from llm_providers import MockLLM

llm = MockLLM(seed=42)
print('MockLLM ready ✅')

> 🔌 **Using a real provider (OpenAI / Anthropic / Gemini / Ollama).**
> Every notebook in this module uses an offline `MockLLM` by default so you can run them without internet or API keys. Once you want real intelligence in the answers, swap one line — the unified interface in [`llm_providers.py`](../llm_providers.py) lets you use **OpenAI**, **Anthropic** (Claude), **Google** (Gemini), or a **local model via Ollama** without changing anything else.
> See the [LLM Providers Guide](../06_ai_engineering/A1_llm_providers_guide.ipynb) for the swap-in instructions, model recommendations, and cost estimates.


## 3. Your first prompt — system + user messages

Now we hand our intern their first note. Modern LLM APIs use a **chat-style** format with a list of messages. Each message has a **role** and **content**:

- `"system"` — the *instructions*. Sets the model's job, tone, and constraints. Sent once. (This is the *standing brief* you'd pin above the intern's desk.)
- `"user"` — the actual input you want the model to act on. (Today's specific task.)
- `"assistant"` — the model's previous reply (for multi-turn conversations).

The classic mistake of beginners is to cram everything into the user message. Instead, treat the system message as the **stable instructions** and the user message as the **variable input** — like a function and its arguments. Below, we ask the intern to summarise a piece of feedback from our inbox.

In [ ]:
response = llm.chat(
    messages=[
        {"role": "system", "content":
            "You are a helpful assistant that summarises customer feedback in one sentence."},
        {"role": "user",   "content":
            "I love the new dashboard! It is so much faster and the dark mode is amazing. "
            "The mobile app still has some sync issues but overall it is a huge improvement."},
    ],
    model="mock-mini",
    temperature=0.0,
)

print("Reply :", response["text"])
print("Model :", response["model"])
print(f"Tokens: in={response['tokens_in']}, out={response['tokens_out']}")


> 💡 **Temperature.** A number between 0 and ~1 that controls randomness. Use `temperature=0` for classification, extraction, or anything you want to be *deterministic*. Use a slightly higher value (0.3–0.7) for creative tasks like rephrasing or brainstorming.

### 🔬 What actually happens: the model is STATELESS

You just sent a `messages` list and got one string back. Here is the part that trips up
almost everyone: **the model has no memory.** Each `chat()` call is a *pure function* of
the list you send — same list in, same kind of answer out. There is no hidden "conversation"
living inside the model between calls.

So what is "conversation memory"? It is just **you re-sending the whole growing list every
turn**. The model re-reads the entire history from scratch each time:

```text
Turn 1                Turn 2                       Turn 3
─────────             ──────────────────           ─────────────────────────────
[ system ]            [ system ]                   [ system ]
[ user 1 ]    ───►    [ user 1 ]                   [ user 1 ]
                      [ assistant 1 ]   ───►       [ assistant 1 ]
   (2 msgs)            [ user 2 ]                    [ user 2 ]
                                                    [ assistant 2 ]
                         (4 msgs)                   [ user 3 ]
                                                       (6 msgs)
   send these ─► get reply ─► APPEND reply + next user msg ─► send the bigger list again
```

The list only grows. That is *why long chats cost more*: every turn re-sends every earlier
message, so `tokens_in` climbs turn after turn. **YOU hold the memory, not the model.**

> 🧠 **Mental model.** A chat LLM is a pure function `f(messages) -> reply`. The model is a
> stateless calculator over the list you hand it; the *conversation* lives in **your** list,
> which you grow yourself by appending each `user` turn and each `assistant` reply.


In [ ]:
# Proof — a deterministic OFFLINE mock. NO API, stdlib only.
# It just counts how many messages it was sent: same input -> same output, no memory.
def fake_llm(messages):
    """A pure, stateless 'LLM': reply depends ONLY on the messages passed in."""
    n_user = sum(1 for m in messages if m["role"] == "user")
    last   = messages[-1]["content"]
    # The reply is a deterministic function of the list — nothing is remembered between calls.
    return f"(saw {len(messages)} msgs, {n_user} from you) you just said: {last!r}"

# Call it TWICE with the SAME list -> identical answer. That is what 'stateless' means.
probe = [
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user",   "content": "Hello!"},
]
print(fake_llm(probe))
print(fake_llm(probe))     # exact same output — no hidden state changed
print("Equal twice? ->", fake_llm(probe) == fake_llm(probe))


In [ ]:
# Now a real (tiny) conversation. THE LIST is the memory — we grow it ourselves.
messages = [{"role": "system", "content": "You are a helpful assistant."}]

def turn(messages, user_text):
    """One conversational turn: append user msg, call the model, append the reply."""
    messages.append({"role": "user", "content": user_text})   # 1) you add your message
    reply = fake_llm(messages)                                 # 2) send the WHOLE list
    messages.append({"role": "assistant", "content": reply})   # 3) you store the reply
    return reply

for user_text in ["What's 2+2?", "And times 10?", "Thanks!"]:
    turn(messages, user_text)
    print(f"after asking {user_text!r:18} -> messages list now has {len(messages)} entries")

print("\nFinal conversation (this list IS the 'memory'):")
for m in messages:
    print(f"  {m['role']:<9}: {m['content'][:60]}")


Notice the list grew **2 entries per turn** (your `user` message + the model's
`assistant` reply), and every call re-sent the *entire* history. The model never
remembered anything — `messages` did. That single idea explains three things you'll
rely on for the rest of this module:

- **Why chats cost more over time** — each turn re-sends all prior messages, so input
  tokens grow turn by turn.
- **Why you can edit/trim history** — drop or summarise old turns and the next call simply
  sees a different list (this is "context window management").
- **Why `MockLLM` and a real API are interchangeable** — both are just `f(messages) -> reply`.

> ⚠️ Because nothing is remembered server-side, forgetting to **append the assistant reply**
> is a classic bug: the model then never "sees" what it previously said, and the conversation
> loses coherence.


## 4. The four core prompt patterns

A vague note gets vague work back — from an intern *or* an LLM. So before we point this at the whole inbox, let's learn to write notes that get reliable results. You'll see hundreds of "prompt engineering" tips on the internet. They almost all boil down to **four patterns**.

### Pattern 1 — Clear instructions

The single biggest lever. Be explicit about:

1. **Role** — *"You are a senior support analyst."*
2. **Task** — *"Classify the customer's sentiment."*
3. **Constraints** — *"Reply with exactly one of: positive, neutral, negative."*

Vague prompts produce vague output. Boring as that sounds, it is what works.

In [ ]:
# A *good* classification prompt: role + task + constraint
SYSTEM_PROMPT = (
    "You are a customer-experience analyst. "
    "Classify the sentiment of the customer message as exactly one of: "
    "positive, neutral, negative. Reply with only the label — no extra words."
)

def classify_sentiment(text):
    r = llm.chat(
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": text},
        ],
        temperature=0.0,
    )
    return r["text"].strip().lower()


for msg in [
    "Absolutely love the latest update — it saved me hours!",
    "The app crashed three times today. I'm so frustrated.",
    "The form has 12 fields. Some of them are useful.",
]:
    print(f"  {classify_sentiment(msg):<8}  ←  {msg[:60]}")


### Pattern 2 — Few-shot examples

Instead of *describing* the task in words, *show* the model two or three examples. Especially useful when:

- the task is hard to describe precisely,
- you want a very specific output format,
- the model keeps drifting away from your desired style.

The shape is always: `instructions` → `example 1` → `example 2` → … → `the real input`.

In [ ]:
FEW_SHOT_SYSTEM = (
    "You classify the topic of customer messages. "
    "Choose exactly one of: billing, tech, feature, other. "
    "Reply with only the label."
)

# A few examples wrapped as user/assistant turns
few_shot_messages = [
    {"role": "system",    "content": FEW_SHOT_SYSTEM},
    {"role": "user",      "content": "I was charged twice for my subscription last month."},
    {"role": "assistant", "content": "billing"},
    {"role": "user",      "content": "Could you add dark mode to the mobile app?"},
    {"role": "assistant", "content": "feature"},
    {"role": "user",      "content": "I can't log in — the password reset link is broken."},
    {"role": "assistant", "content": "tech"},
]

def classify_topic(text):
    msgs = few_shot_messages + [{"role": "user", "content": text}]
    r = llm.chat(messages=msgs, temperature=0.0)
    return r["text"].strip().lower()


for msg in [
    "My invoice doesn't match my plan — please refund the difference.",
    "I keep getting a 500 error when uploading a file.",
    "Would love it if you supported CSV export.",
]:
    print(f"  {classify_topic(msg):<8}  ←  {msg}")


### Pattern 3 — Structured (JSON) output

This is the **single most useful pattern for automation**. If you ask the model to "reply in JSON with keys X, Y, Z" you can `json.loads()` the reply and use it like any other Python data.

You'll do this every time you build an AI feature that hands its result to another step in a pipeline.

In [ ]:
STRUCTURED_SYSTEM = (
    "You analyse customer messages. "
    "Return a JSON object with exactly two keys: "
    "  - 'sentiment': one of positive | neutral | negative\n"
    "  - 'topic':     one of billing | tech | feature | other\n"
    "Reply with the JSON only. No prose, no markdown fences."
)

def analyse(text):
    r = llm.chat(
        messages=[
            {"role": "system", "content": STRUCTURED_SYSTEM},
            {"role": "user",   "content": text},
        ],
        temperature=0.0,
    )

    # The model's reply is a JSON string — parse it.
    try:
        return json.loads(r["text"])
    except json.JSONDecodeError:
        # Always have a fallback for when the model misbehaves
        return {"sentiment": "unknown", "topic": "unknown", "_raw": r["text"]}


result = analyse("My password reset link is broken and I can't log in. Please help!")
print(result)
print(f"\nSentiment: {result['sentiment']!r}, Topic: {result['topic']!r}")


> ⚠️ **Always wrap `json.loads` in `try / except`.** Real LLMs occasionally include markdown fences (```` ```json ... ``` ````), trailing prose, or a typo. Defensive parsing is non-negotiable in production.

### Pattern 4 — Chain-of-thought (CoT)

For multi-step reasoning, ask the model to think *step by step* before giving the final answer. The phrasing literally is: *"First, list the steps. Then give the answer."* It works because the model has more output tokens to spend on intermediate reasoning steps.

We won't dwell on it here — the mock isn't smart enough to demonstrate — but remember the trick. When the model gets an answer wrong on a reasoning-heavy task, add "let's think step by step" and try again.

---

### ✋ Quick exercise (~2 min) — Write a one-label classifier prompt

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Triage needs a new label. Using the **Pattern 1** recipe (role + task + constraint), write a `PRIORITY_PROMPT` and a `classify_priority(text)` function that returns **exactly one of** `high`, `normal`, `low` — and nothing else. Model it on `classify_sentiment` from earlier.

```python
PRIORITY_PROMPT = ...  # role + task + constraint, "reply with only the label"
```

In [ ]:
# ✍️ Your turn 👇
# Following the Pattern 1 recipe (role + task + constraint), write PRIORITY_PROMPT
# and a classify_priority(text) function that returns only the label.
PRIORITY_PROMPT = ...

def classify_priority(text):
    ...

# Try it once you've filled it in:
# print(classify_priority("The login is completely broken and I can't access my account!"))

<details>
<summary>✅ <b>Solution</b></summary>

```python
PRIORITY_PROMPT = (
    "You are a support triage analyst. "
    "Classify the urgency of the customer message as exactly one of: "
    "high, normal, low. Reply with only the label — no extra words."
)

def classify_priority(text):
    r = llm.chat(
        messages=[
            {"role": "system", "content": PRIORITY_PROMPT},
            {"role": "user",   "content": text},
        ],
        temperature=0.0,
    )
    return r["text"].strip().lower()

print(classify_priority("The login is completely broken and I can't access my account!"))
```

This reuses the **Pattern 1** recipe (role + task + constraint) and the same `llm.chat` call shape as `classify_sentiment` — only the system prompt and label set change.
</details>

## 5. Batch processing — classify a whole inbox

Here's the payoff we've been building toward: the **overnight feedback inbox** from the top of the notebook. One well-written note (prompt) plus a Python loop, and our forgetful intern handles fifty messages while you sleep. This is where Python loops + LLM calls really shine. Suppose you have 50 fresh pieces of customer feedback overnight. You want a sentiment, a topic, and a one-line summary for each.

In [ ]:
# A small but realistic batch of customer feedback
feedback_batch = [
    "I love the new dashboard! It's much faster and the dark mode is amazing.",
    "The app crashed three times today. I'm so frustrated with this bug.",
    "Could you please add CSV export? I really need it for my reports.",
    "My invoice doesn't match my plan — please refund the difference of $45.",
    "Everything works fine, no complaints, no praise either.",
    "The login is broken after the latest update. I can't access my account.",
    "Fantastic support team, they resolved my issue in minutes. Thank you!",
    "Why is the renewal price so high? I'm considering cancelling my subscription.",
    "The new feature is great but the loading is really slow on mobile.",
    "Please add support for Markdown in the editor. It would be a huge help.",
]
print(f"Batch size: {len(feedback_batch)}")


In [ ]:
# Run all of them through the structured-output analyser
SYSTEM = (
    "You analyse customer messages. "
    "Return JSON with keys 'sentiment' (positive/neutral/negative) and "
    "'topic' (billing/tech/feature/other). "
    "Reply with the JSON only."
)

def analyse_and_summarise(text):
    # 1) Sentiment + topic in one structured call
    r1 = llm.chat(
        messages=[{"role": "system", "content": SYSTEM},
                  {"role": "user",   "content": text}],
        temperature=0.0,
    )
    try:
        labels = json.loads(r1["text"])
    except json.JSONDecodeError:
        labels = {"sentiment": "unknown", "topic": "unknown"}

    # 2) A short summary in a second call
    r2 = llm.chat(
        messages=[{"role": "system", "content": "Summarise the customer message in one sentence."},
                  {"role": "user",   "content": text}],
        temperature=0.0,
    )
    labels["summary"] = r2["text"].strip()
    labels["tokens_in"]  = r1["tokens_in"]  + r2["tokens_in"]
    labels["tokens_out"] = r1["tokens_out"] + r2["tokens_out"]
    return labels


# Process the batch
results = [analyse_and_summarise(msg) for msg in feedback_batch]

# Show first few results
for msg, res in list(zip(feedback_batch, results))[:5]:
    print(f"[{res['sentiment']:<8} / {res['topic']:<7}]  {res['summary']}")


### A KPI report on the batch

Once each item has structured fields, aggregating is just Python — exactly the techniques from Notebooks 2 and 3.

In [ ]:
from collections import Counter

n = len(results)
sentiments = Counter(r["sentiment"] for r in results)
topics     = Counter(r["topic"]     for r in results)

tokens_in_total  = sum(r["tokens_in"]  for r in results)
tokens_out_total = sum(r["tokens_out"] for r in results)

price_in_per_1k  = 0.0006
price_out_per_1k = 0.0024
cost = (tokens_in_total / 1000 * price_in_per_1k +
        tokens_out_total / 1000 * price_out_per_1k)

print(f"📊 Feedback batch — {n} messages")
print("-" * 40)
print("Sentiment mix:")
for label, count in sentiments.most_common():
    print(f"  {label:<10}: {count:>2}  ({count/n:.0%})")
print("\nTopic mix:")
for label, count in topics.most_common():
    print(f"  {label:<10}: {count:>2}  ({count/n:.0%})")
print(f"\nTokens in : {tokens_in_total:,}")
print(f"Tokens out: {tokens_out_total:,}")
print(f"Est. cost : ${cost:.4f}")


### The same KPIs as a picture

`results` is a list of dicts with consistent keys — exactly what `pd.DataFrame` was built for. One conversion, and the whole pandas + plotting toolkit from Notebooks 6–7 applies to your LLM output. This is the punchline of the pipeline: **unstructured text in → ordinary analytics out.**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

report = pd.DataFrame(results)
display(report[["sentiment", "topic", "tokens_in", "tokens_out"]].head())

fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
report["sentiment"].value_counts().plot(kind="barh", ax=axes[0], color="steelblue",
                                        title="Sentiment mix", xlabel="messages")
report["topic"].value_counts().plot(kind="barh", ax=axes[1], color="seagreen",
                                    title="Topic mix", xlabel="messages")
fig.tight_layout()

**Stop and notice what just happened.** We turned a 10-item pile of unstructured text into a structured table of sentiments, topics and summaries, *plus* a cost report — in under 30 lines of code. That is the entire point of AI-assisted automation: turn the messy stuff at the edges of your business into clean rows you can act on.

---

### ✋ Quick exercise (~2 min) — Triage the negative messages

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

The morning team only wants to read the unhappy customers first. Using the `results` list the batch just produced (a dict per message), build `negative_summaries` — a list of the `summary` text for every message whose `sentiment` is `"negative"`. No new LLM calls; the structured fields are already there.

In [ ]:
# ✍️ Your turn 👇
# `results` already holds a dict per message with keys: sentiment, topic, summary, ...
# Build a list of the one-line summaries for every NEGATIVE message — no new LLM calls.
negative_summaries = ...

print(negative_summaries)

<details>
<summary>✅ <b>Solution</b></summary>

```python
negative_summaries = [r["summary"] for r in results if r["sentiment"] == "negative"]

for s in negative_summaries:
    print("-", s)
print(f"\n{len(negative_summaries)} negative messages need attention.")
```

Because every item in `results` is a dict with consistent keys, a single list comprehension filters by `sentiment` and pulls the `summary` — no extra LLM calls needed, since the batch already produced the structured fields.
</details>

## 6. A tiny retrieval workflow

Our intern is great at sorting feedback — but what happens when a customer *asks a question* about your refund policy? The intern has never read your docs, so they'd confidently make something up. The fix is the same one you'd use for a real intern: **put the relevant page on their desk before they answer.**

What if you want the model to answer questions **about your own data** — a knowledge base, your product docs, your company policies?

The technique is called **Retrieval-Augmented Generation (RAG)**:

```
question  ─┐
            ├──→ retrieve top-k relevant snippets ──→ LLM ──→ answer
your docs ─┘
```

In a production system the retrieval step uses **embeddings** (vector search). For learning the pattern, a simple **keyword overlap** scorer is enough — and works without any extra dependencies.

In [ ]:
# A tiny in-memory "knowledge base"
docs = [
    ("billing.refunds",
     "Refunds are issued within 5 business days. Customers can request a refund "
     "from the Billing page if the request is within 30 days of purchase."),

    ("billing.subscriptions",
     "Subscriptions renew automatically. To cancel, go to Settings → Billing "
     "and click 'Cancel subscription'. You will retain access until the period ends."),

    ("tech.password",
     "If you can't log in, use the 'Forgot password' link on the sign-in page. "
     "Reset emails arrive within 5 minutes; check your spam folder if missing."),

    ("tech.errors",
     "A 500 error usually indicates a temporary backend issue. Wait 60 seconds "
     "and retry. If the error persists, contact support with the trace ID shown."),

    ("feature.exports",
     "CSV and JSON exports are available on Pro plans. Use the 'Export' button "
     "in the top-right corner of any report. XLSX export is on the roadmap."),
]
print(f"Knowledge base has {len(docs)} documents.")


In [ ]:
def keyword_score(query, doc_text):
    """Number of distinct query words that appear in the document."""
    q_words = set(re.findall(r'[a-zA-Z]+', query.lower()))
    d_words = set(re.findall(r'[a-zA-Z]+', doc_text.lower()))
    return len(q_words & d_words)


def retrieve(query, docs, k=2):
    scored = [(keyword_score(query, text), name, text) for name, text in docs]
    scored.sort(reverse=True)            # highest score first
    return [(name, text) for score, name, text in scored[:k] if score > 0]


# Try it
query = "How do I get a refund for my subscription?"
top = retrieve(query, docs, k=2)
for name, text in top:
    print(f"  → {name}\n     {text}\n")


In [ ]:
def rag_answer(question, docs):
    snippets = retrieve(question, docs, k=2)
    if not snippets:
        context = "(no relevant snippets found)"
    else:
        context = "\n\n".join(f"[{name}] {text}" for name, text in snippets)

    system = (
        "You answer the user's question using ONLY the context provided. "
        "If the context does not contain the answer, say so clearly."
    )
    user = f"Context:\n{context}\n\nQuestion: {question}"

    r = llm.chat(
        messages=[{"role": "system", "content": system},
                  {"role": "user",   "content": user}],
        temperature=0.0,
    )
    return r["text"], snippets


answer, used = rag_answer("How do I cancel my subscription?", docs)
print("Answer:", answer)
print("\nSources used:")
for name, _ in used:
    print(f"  - {name}")


> 🎯 **What's the win here?** Without retrieval, the model would have to *guess* your refund policy. With retrieval, it answers from your *actual* policy — so you can deploy the same model across very different domains by just swapping the knowledge base.

> 💡 **In production** you'd replace `keyword_score` with vector embeddings (e.g. `text-embedding-3-small`) plus a vector store. The control-flow shape — retrieve → assemble context → call the model — stays *identical*.

---

### ✋ Quick exercise (~2 min) — Predict the retrieved doc

> 🧑‍🏫 *Live checkpoint — pause and try this before peeking at the solution.*

Before running it, **guess** which document the keyword retriever will return for the question below — then call `retrieve(question, docs, k=1)` to check. Which doc name comes back, and which shared words earned it the top score?

```python
question = "Why do I get a 500 error when I upload a file?"
```

In [ ]:
# ✍️ Your turn 👇
# Use the retrieve() function on the knowledge base `docs`.
# Which single doc does keyword overlap pick for this question?
question = "Why do I get a 500 error when I upload a file?"
top = ...        # call retrieve(...) with k=1

print(top)

<details>
<summary>✅ <b>Solution</b></summary>

```python
question = "Why do I get a 500 error when I upload a file?"
top = retrieve(question, docs, k=1)
print(top[0][0])   # -> 'tech.errors'
```

`retrieve` ranks each doc by `keyword_score` (count of shared words). The `tech.errors` doc shares words like *error*, *500*, and *retry*, giving it the highest overlap — so it surfaces first even though we never told `retrieve` anything about error codes.
</details>

## 7. Cost, latency, evaluation — the engineer's checklist

You'd never let a new intern's work go straight to customers unchecked — and you shouldn't ship an LLM feature unchecked either. "Cool, it works on my five examples" is *not* a production system. Before you ship, run a few back-of-the-envelope numbers and spot-check the intern's accuracy.

In [ ]:
# A small evaluation set with known correct labels
eval_set = [
    ("I love the new dark mode!",                                "positive"),
    ("Crashed again. Furious.",                                  "negative"),
    ("Could you add a CSV export?",                              "neutral"),     # a request, not a sentiment
    ("Refund requested — wrong charge on invoice.",              "negative"),
    ("Works fine, no issues.",                                   "positive"),    # neutral-ish too — tricky
]

correct = 0
total_tokens_in  = 0
total_tokens_out = 0
total_latency_s  = 0.0

for text, expected in eval_set:
    r = llm.chat(
        messages=[
            {"role": "system", "content":
                "Classify the sentiment as positive, neutral, or negative. Reply with only the label."},
            {"role": "user", "content": text},
        ],
        temperature=0.0,
    )
    pred = r["text"].strip().lower()
    ok   = pred == expected
    correct += int(ok)
    total_tokens_in  += r["tokens_in"]
    total_tokens_out += r["tokens_out"]
    total_latency_s  += r["latency_s"]
    mark = "✅" if ok else "❌"
    print(f"  {mark}  expected={expected:<8}  got={pred:<8}  ←  {text[:50]}")

# Headline numbers
n = len(eval_set)
acc = correct / n
cost = (total_tokens_in / 1000 * 0.0006 +
        total_tokens_out / 1000 * 0.0024)
print(f"\nAccuracy : {acc:.0%}  ({correct}/{n})")
print(f"Latency  : {total_latency_s*1000:.1f} ms total ({total_latency_s/n*1000:.1f} ms / call)")
print(f"Cost     : ${cost:.4f} for the whole eval set")


**A useful habit.** Every AI feature you ship should have a small "regression set" like this — 50–500 hand-labelled examples you can re-run in 30 seconds whenever you change a prompt, a model, or a temperature. Without it, every prompt tweak is a gamble.

## 8. Going live with a real model

Everything you've written so far works with *any* chat-style LLM. To swap the mock for a real provider, you change one function — `llm.chat()` — and your application code stays the same.

Here are the **shapes** of the two most common SDKs. **You should NOT run these cells in this notebook** unless you have a key set up — they are here as a reference you can copy.

### Pattern: OpenAI

```python
# pip install openai
# export OPENAI_API_KEY=...

from openai import OpenAI
client = OpenAI()                            # picks up OPENAI_API_KEY from env

def chat(messages, model="gpt-4o-mini", temperature=0.0):
    resp = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
    )
    return {
        "text":       resp.choices[0].message.content,
        "model":      resp.model,
        "tokens_in":  resp.usage.prompt_tokens,
        "tokens_out": resp.usage.completion_tokens,
    }
```

### Pattern: Anthropic

```python
# pip install anthropic
# export ANTHROPIC_API_KEY=...

import anthropic
client = anthropic.Anthropic()               # picks up ANTHROPIC_API_KEY from env

def chat(messages, model="claude-haiku-4-5-20251001", temperature=0.0):
    # Anthropic puts the system message in a separate parameter
    system = next((m["content"] for m in messages if m["role"] == "system"), "")
    user_assistant = [m for m in messages if m["role"] != "system"]

    resp = client.messages.create(
        model=model,
        system=system,
        max_tokens=1024,
        temperature=temperature,
        messages=user_assistant,
    )
    return {
        "text":       resp.content[0].text,
        "model":      resp.model,
        "tokens_in":  resp.usage.input_tokens,
        "tokens_out": resp.usage.output_tokens,
    }
```

> 🔐 **Never paste API keys into a notebook.** Read them from environment variables (`os.getenv("OPENAI_API_KEY")`) or a `.env` file plus `python-dotenv`. Keys in notebooks tend to end up in git history.

> 💡 The function-call shape is the same. That is why we built `MockLLM.chat` with the same signature — the rest of your pipeline doesn't care which engine answers.

## 🧪 Practice exercises

### Exercise 1 — ⭐ A safer JSON parser

Write a function `safe_json(text)` that:

1. Strips markdown code fences (```` ```json ... ``` ````) if present.
2. Returns the parsed object on success.
3. Returns `None` (not a crash) on `JSONDecodeError`.

Then test it on these three strings:

```python
'{"a": 1}'
'```json\n{"a": 1}\n```'
'this is not JSON'
```

In [ ]:
# Your code here  👇
def safe_json(text):
    pass


<details>
<summary>💡 <b>Solution</b></summary>

```python
import json, re

def safe_json(text):
    if text is None:
        return None
    # Drop markdown fences like ```json ... ``` or ``` ... ```
    cleaned = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip(), flags=re.IGNORECASE | re.MULTILINE)
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        return None

print(safe_json('{"a": 1}'))                            # {'a': 1}
print(safe_json('```json\n{"a": 1}\n```'))             # {'a': 1}
print(safe_json("this is not JSON"))                     # None
```

**Why this matters.** Every batch you run will have *some* malformed responses. Defensive parsing keeps one bad reply from killing the whole job.
</details>

### Exercise 2 — ⭐⭐ Classify and split

Given the batch below, build two lists: one of messages classified as `"negative"`, one of everything else. Use the `analyse` function we built earlier — or write your own one-line version with `classify_sentiment`.

In [ ]:
batch_b = [
    "Great product, would recommend!",
    "The app keeps crashing on launch.",
    "Pricing page is unclear.",
    "Customer support resolved my issue quickly. Thanks!",
    "Refund taking forever. Very disappointed.",
]

# Your code here  👇
negatives = []
others    = []


<details>
<summary>💡 <b>Solution</b></summary>

```python
negatives = []
others    = []

for msg in batch_b:
    if classify_sentiment(msg) == "negative":
        negatives.append(msg)
    else:
        others.append(msg)

print("Negatives:")
for m in negatives:
    print("  -", m)
print("\nOthers:")
for m in others:
    print("  -", m)
```

**More Pythonic (one-liner):**

```python
negatives = [m for m in batch_b if classify_sentiment(m) == "negative"]
others    = [m for m in batch_b if classify_sentiment(m) != "negative"]
```

Note the comprehension version calls the LLM twice per message. For long batches, prefer the explicit loop so each message is classified once.
</details>

### Exercise 3 — ⭐⭐ Add a confidence band to the RAG answer

Modify `rag_answer` to also return a *confidence band* based on the number of high-scoring snippets retrieved:

- **high** — at least 2 snippets each with a score ≥ 2
- **medium** — at least 1 snippet with a score ≥ 2
- **low** — otherwise

This lets a downstream system decide whether to show the AI answer or escalate to a human.

In [ ]:
# Your code here  👇
def rag_answer_with_confidence(question, docs):
    pass


<details>
<summary>💡 <b>Solution</b></summary>

```python
def rag_answer_with_confidence(question, docs):
    scored = [(keyword_score(question, text), name, text) for name, text in docs]
    scored.sort(reverse=True)
    snippets = [(name, text) for score, name, text in scored[:3] if score > 0]
    high_scores = [score for score, *_ in scored[:3] if score >= 2]

    if len(high_scores) >= 2:
        confidence = "high"
    elif len(high_scores) >= 1:
        confidence = "medium"
    else:
        confidence = "low"

    if not snippets:
        return {"answer": "(no relevant snippets found)", "confidence": "low", "sources": []}

    context = "\n\n".join(f"[{name}] {text}" for name, text in snippets)
    r = llm.chat(
        messages=[
            {"role": "system", "content":
                "You answer the user's question using ONLY the context provided."},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"},
        ],
        temperature=0.0,
    )
    return {
        "answer":     r["text"],
        "confidence": confidence,
        "sources":    [name for name, _ in snippets],
    }


result = rag_answer_with_confidence("How do I cancel my subscription?", docs)
print(result)
```

**Why this matters.** A confidence signal lets you build *graceful degradation* into AI features — show the answer when the retrieval is solid, ask a human when it isn't. That single field can be the difference between a feature your users trust and one they don't.
</details>

## 🧠 Stretch exercises

Two more applied exercises to deepen the material. Try them yourself before opening the solution.


### Stretch exercise C — ⭐⭐⭐ Few-shot prompt with three examples

Build a classification prompt that shows the model three labelled examples (a few-shot prompt) before asking it to classify a new message. Use the MockLLM you imported at the top.

Classify customer messages into one of: `billing`, `tech`, `feature`, `other`.

In [ ]:
# Your code here  👇
EXAMPLES = [
    ("My invoice is wrong",       "billing"),
    ("App crashes on startup",    "tech"),
    ("Add a dark mode please",    "feature"),
]
new_msg = "Where is my refund?"

# Build a system prompt that shows the examples, then ask the model to classify new_msg.
# Use llm.chat(messages=[...]) — llm is the MockLLM defined earlier in this notebook.
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
EXAMPLES = [
    ("My invoice is wrong",       "billing"),
    ("App crashes on startup",    "tech"),
    ("Add a dark mode please",    "feature"),
]
new_msg = "Where is my refund?"

shots = "\n".join(f"- {m!r} → {label}" for m, label in EXAMPLES)
system = (
    "Classify the topic of each customer message into one of: "
    "billing, tech, feature, other.\n\n"
    "Examples:\n" + shots
)

resp = llm.chat(messages=[
    {"role": "system", "content": system},
    {"role": "user",   "content": new_msg},
])
print(resp["text"])
```

**Reasoning.** Few-shot prompting is the cheapest behavioural lever you have on an LLM. Three rules of thumb. (1) **Show, then constrain.** The examples come first; the rule ("one of: billing, tech, feature, other") sits in the system prompt so the model knows the output space. (2) Use a **consistent example format** — input arrow output, one per line — so the model pattern-matches the format too. (3) On the MockLLM you'll get back the topic word; on a real model you'd also pin `temperature=0` and validate the output is in your allowed set.
</details>

### Stretch exercise D — ⭐⭐⭐ Validate JSON output with retry

Ask the model for a JSON object with keys `sentiment` and `topic`. Parse the response; if it isn't valid JSON, **retry once** with a stricter system prompt. Print either the parsed dict or the failure reason.

In [ ]:
# Your code here  👇
import json

def ask_json(user_msg, strict=False):
    sys = ("Reply with strict JSON only. " if strict else "") + \
          "Classify the message into JSON with keys sentiment (positive/negative/neutral) " \
          "and topic (billing/tech/feature/other)."
    return llm.chat(messages=[
        {"role": "system", "content": sys},
        {"role": "user",   "content": user_msg},
    ])["text"]

user_msg = "My refund was supposed to arrive last week"
# ...


<details>
<summary>💡 <b>Solution — click to expand</b></summary>

```python
import json

def ask_json(user_msg, strict=False):
    sys = ("Reply with strict JSON only. " if strict else "") + \
          "Classify the message into JSON with keys sentiment (positive/negative/neutral) " \
          "and topic (billing/tech/feature/other)."
    return llm.chat(messages=[
        {"role": "system", "content": sys},
        {"role": "user",   "content": user_msg},
    ])["text"]

def parse_with_retry(user_msg, max_retries=1):
    raw = ask_json(user_msg, strict=False)
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        if max_retries == 0:
            return {"_error": "invalid JSON after retry", "raw": raw}
        raw = ask_json(user_msg, strict=True)
        try:
            return json.loads(raw)
        except json.JSONDecodeError:
            return {"_error": "invalid JSON after retry", "raw": raw}

print(parse_with_retry("My refund was supposed to arrive last week"))
```

**Reasoning.** The most common production failure mode of LLMs that 'output JSON' is **almost-JSON** — extra prose, a trailing comma, backticks. Three habits this exercise builds. (1) **Validate, don't trust.** Call `json.loads` and react to the exception. (2) **Retry with a stricter prompt.** The cheapest mitigation is just to ask again with `"Reply with strict JSON only."` — modern models comply most of the time. (3) **Cap the retries**, return a structured error on giving up. Real apps would also clamp `temperature=0` and run the output through a JSON schema (e.g. with Pydantic) before downstream code sees it.
</details>

## 🧠 Key takeaways

We started the night with a messy inbox and a brilliant-but-forgetful intern. We end it with a script that reads, sorts, summarises, and reports — and a checklist for trusting its work. Here's what made that possible:

1. **LLM calls are function calls.** Treat them with the same engineering hygiene as any third-party API — typed inputs, structured outputs, retries, costs.
2. **System ≠ user.** Stable instructions (the intern's standing brief) go in the system message; variable input (today's task) goes in the user message.
3. **Four prompt patterns cover most of what you need:** clear instructions, few-shot examples, structured (JSON) output, and chain-of-thought.
4. **Structured output is the bridge to automation.** Once the model returns JSON, the rest of your code is just Python.
5. **Batch + aggregate** beats one-off prompts: classify the inbox, count the topics, report on the cost.
6. **Retrieval grounds the model in your data** — put the right page on the intern's desk before they answer. Even a one-page RAG using keyword overlap is a real, useful system.
7. **Always wrap `json.loads` in `try/except`.** Your intern occasionally hands back sloppy work.
8. **Track cost, latency, accuracy** with a small evaluation set. A 30-second regression test pays for itself within a week.
9. **Never put API keys in notebooks.** Use environment variables.

> 🧭 **The mental model, one more time.** Everything here was about managing a forgetful intern: clear notes (prompts), everything on the page (no memory — *you* hold the conversation), a fileable format (JSON), and spot-checks (evaluation). When a real LLM project misbehaves, ask "how would I have managed an intern here?" — the fix is usually the same.

## ✅ Self-assessment

- [ ] Explain why the system prompt should be the instructions and the user prompt the input
- [ ] Write a prompt that returns a single label, deterministically
- [ ] Write a prompt that returns a JSON object you can parse
- [ ] Run a batch classification and produce a KPI summary
- [ ] Build a tiny RAG pipeline (retrieve → assemble context → call the model)
- [ ] Estimate cost from token counts for a given price per 1K
- [ ] Swap `MockLLM` for `openai` or `anthropic` without touching the rest of your code

## 🎓 Where to go from here

- **Embeddings & vector stores** — replace keyword retrieval with semantic search.
- **Tool / function calling** — let the model call your functions, not just answer in text.
- **Agents** — multi-step reasoning loops that plan, act, and observe.
- **Evaluation frameworks** — automated scoring beyond exact-match accuracy.

Each of these is a notebook of its own. You now have the Python fluency and the AI-engineering vocabulary to read those notebooks and *follow what's going on* — which is more than half the battle.

## 🚀 Next step

Continue with **Notebook 11 (fast track) — Embeddings and Semantic Retrieval**, where the keyword matching from Section 6 grows up: instead of putting pages on the intern's desk by *word overlap*, we'll retrieve them by *meaning*.